<div align="center"> 

# VIDEO QA POLITIC FACT-CHECK BOT

</div>

##### My pipeline 
1. Extract transcript from YouTube videos
2. Organize and clean the dataset
3. Build the RAG pipeline
4. Connect integrations (models, vector DB)
5. Generate embeddings
6. Optional local embedding models
7. Tokenization and chunking
8. Store embeddings in a vector database
9. Manage API keys securely
10. Build the chatbot interface
#  Modificareeeee!!!!

## 1 Setup Environment (libraries & imports)

#### Install required libraries: 

In [ ]:
# =========================================================
# 1️⃣ YouTube transcript extraction
# Download transcripts from YouTube videos
# =========================================================
!pip install youtube-transcript-api


# =========================================================
# 2️⃣ Data manipulation
# Organize transcripts, clean text, export datasets
# =========================================================
!pip install pandas




# =========================================================
# 4️⃣ LangChain integrations
# External integrations (vector DBs, tools, etc.)
# =========================================================
!pip install langchain-community


# =========================================================
# 5️⃣ LangChain text splitting (NEW PACKAGE)
# Required for RecursiveCharacterTextSplitter
# =========================================================
!pip install langchain-text-splitters


# =========================================================
# 6️⃣ OpenAI integration
# Used for embeddings and chat models
# =========================================================
!pip install langchain-openai openai


# =========================================================
# 7️⃣ Vector database
# Store embeddings and enable semantic search
# =========================================================
!pip install chromadb


# =========================================================
# 8️⃣ Embedding models (optional local embeddings)
# Alternative to OpenAI embeddings
# =========================================================
!pip install sentence-transformers


# =========================================================
# 9️⃣ Tokenization utilities
# Used for token counting and chunking
# =========================================================
!pip install tiktoken


# =========================================================
# 🔟 Environment variables
# Manage API keys securely
# =========================================================
!pip install python-dotenv


# =========================================================
# 1️⃣1️⃣ Chatbot web interface
# Optional interface for the final chatbot
# =========================================================
!pip install streamlit

# Installing LangSmith for Second evaluation

!pip install langsmith


In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-openai langchain-text-splitters

In [ ]:
!pip install langchain==0.2.16 \
langchain-core==0.2.38 \
langchain-community==0.2.16 \
langchain-openai==0.1.23 \
langchain-text-splitters==0.2.2

In [ ]:
!pip install -U \
langchain==0.2.16 \
langchain-community==0.2.16 \
langchain-openai==0.1.23 \
langchain-text-splitters==0.2.2 \
chromadb \
youtube-transcript-api \
pandas \
openai \
tiktoken

In [ ]:
# Modification: add Spacey
!pip install spacy
!python -m spacy download en_core_web_sm


#### Import Libraries

In [1]:

# Data handling
import pandas as pd

# YouTube transcript extraction
from youtube_transcript_api import YouTubeTranscriptApi

# NLP preprocessing (tokenization / lemmatization)
import spacy

# Text splitting (chunking)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings model
from langchain_openai import OpenAIEmbeddings

# Vector database
from langchain_community.vectorstores import Chroma

# Document schema
from langchain_core.documents import Document

# LLM model
from langchain_openai import ChatOpenAI

# Tools for the agent
from langchain.tools import tool
from langchain.tools import Tool

# LangChain agent components
from langchain.agents import initialize_agent
from langchain.agents import AgentType

# Memory for conversational context
from langchain.memory import ConversationBufferMemory

# LangSmith monitoring
import os

# We use it to call OpenAI API to generate images with gpt-image-1
from openai import OpenAI



In [3]:
# ==========================================================
# 2. LANGSMITH CONFIGURATION
# ==========================================================
# Enables tracing and monitoring of the pipeline

import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "youtube-qa-bot"


In [5]:
# ==========================================================
# 3. LOAD SPACY NLP MODEL
# ==========================================================
# spaCy is used for NLP preprocessing such as lemmatization

nlp = spacy.load("en_core_web_sm")

In [6]:
### OpenAI API Configuration

from dotenv import load_dotenv
import os

load_dotenv()



True

## 2 Import & Datasets cleaning (YouTube Data Collection)

####  List the YouTube Video IDs

In [7]:
video_ids = [
"Hx_cLr9nnbg",
"wbJuTulibI0",
"pcrbIvyz-Js",
"A2UPjXCf1IU",
"zU9eaxjEgko",
"HGEMscZE5dY",
"YB49-yxy5aA",
"Hvq_qfSJvvY",
"GDATs-o7jgU"
]

In [8]:
# Crate a list where to save all

all_transcripts = []

In [9]:
# We have 2 different possible codes to get our transcriptions

# docs = get_transcripts(video_ids)
# documents = get_youtube_transcripts(video_ids)

In [10]:
# ==========================================================
# DOWNLOAD YOUTUBE TRANSCRIPTS WITH VIDEO METADATA
# ==========================================================
# We save both the transcript text and the video_id
# so we can use metadata later inside the vector database.

all_transcripts = []

for video_id in video_ids:
    try:
        transcript = YouTubeTranscriptApi().fetch(video_id)
        text = " ".join([snippet.text for snippet in transcript])

        text = text.replace("\n", " ").strip()

        all_transcripts.append({
            "video_id": video_id,
            "raw_text": text
        })

        print("Transcript downloaded:", video_id)

    except Exception as e:
        print("Error with video:", video_id)
        print(e)

Transcript downloaded: Hx_cLr9nnbg
Transcript downloaded: wbJuTulibI0
Transcript downloaded: pcrbIvyz-Js
Transcript downloaded: A2UPjXCf1IU
Transcript downloaded: zU9eaxjEgko
Transcript downloaded: HGEMscZE5dY
Transcript downloaded: YB49-yxy5aA
Transcript downloaded: Hvq_qfSJvvY
Transcript downloaded: GDATs-o7jgU


In [11]:
#Let's check if it worked(this will tell u the number of our videos)

len(all_transcripts)

9

In [12]:
# ==========================================================
# MERGE AND CLEAN TRANSCRIPTS
# ==========================================================
# We merge all transcript texts into one large text
# for optional global preprocessing and inspection.

full_text = " ".join([item["raw_text"] for item in all_transcripts])

full_text = full_text.replace("\n", " ")
full_text = full_text.replace("  ", " ")

In [13]:
# ==========================================================
# NLP PREPROCESSING WITH SPACY (LEMMATIZATION)
# ==========================================================
# This function applies NLP preprocessing to text.
# It converts words into lemmas and removes stopwords
# and punctuation.

def preprocess_text(text):

    doc = nlp(text)

    lemmas = [
        token.lemma_.lower()
        for token in doc
        if not token.is_stop and not token.is_punct
    ]

    return " ".join(lemmas)


# Apply preprocessing to the merged transcript text
processed_text = preprocess_text(full_text)

print("Lemmatization completed.")

Lemmatization completed.


In [14]:
# To see the preprocessing

print(processed_text[:1000])

um um group group talk go talk people monarch people queens um rule certain area queen definition rule um consort husband know independent rule go look woman mystic middle ages um people call misfit misfits category exactly woman amazing thing um um know kind fit category actually consider writer time period variety woman let start woman want talk woman name hap sheepit egypt 16th 15th century um um call bc common era ancient period egypt um um follow traditional egypt um rule family marry half brother marry gentleman tomos ii half brother young die seven year marriage serve regent stepson tmos iii um 1489 30 year rule um decide rule um declare king regent anymore unfortunately slide statue particularly standing statue note wear headdress pharaoh male pharaoh wear beard um male pharaoh um put garve male pharaoh um responsible building project uh luxor valley um build mausoleum consider masterpiece ancient architecture um want want mention ne nefertiti u everybody know nefertiti see pic

### We now have our libraries and our merged Youtube datasets

## 2 Text Processing (chunking)

In [15]:
documents = [
    Document(
        page_content=f"Video ID: {item['video_id']}\n\nTranscript:\n{item['raw_text']}",
        metadata={
            "source": "youtube_transcripts",
            "video_id": item["video_id"]
        }
    )
    for item in all_transcripts
]

print(f"{len(documents)} documents created.")

9 documents created.


In [16]:
print(all_transcripts[0])

{'video_id': 'Hx_cLr9nnbg', 'raw_text': "about um are um grouped into three I put them into three groups I talked I going to talk about people who are monarchs that is people who are Queens um and ruled over a certain amount of area to be a queen in my definition here you have to rule over something um not be a consort to a husband but you know have your own have your own independent Rule and then we're going to look at some women who were Mystics in the Middle Ages and um and then go to some people I've called Misfits they're not really Misfits but there's no category exactly to put them into so they're they're just women who did some really amazing things um but um you know they don't kind of quite fit in a category that some of them actually could be considered writers for their time period but the there there are a variety anyway of women so let me start with the first one the first woman I want to talk about is a woman named hap sheepit she's in Egypt in the 16th and into the 15th

In [17]:
# =========================================================
# 3️⃣ Text Splitting (Chunking)
# Split transcripts into smaller chunks for embeddings
# =========================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

Number of chunks: 498


In [18]:
print(chunks[0].metadata)

{'source': 'youtube_transcripts', 'video_id': 'Hx_cLr9nnbg'}


In [19]:
print(chunks[0].page_content[:500])

Video ID: Hx_cLr9nnbg


## 4 Embeddings and Vector Database

In [20]:
# Initialize embedding model

embedding_model = OpenAIEmbeddings()

In [21]:
import shutil
import os

if os.path.exists("./chroma_db"):
    shutil.rmtree("./chroma_db")

In [22]:
# Create vector database from chunks

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="youtube_transcripts",
    persist_directory="./chroma_db"
)

In [23]:
#Verify Databse
vector_db._collection.count()

498

## 5 Retrival Augmented Generation (RAG)

In [43]:
# Create the retriever from the vector database
# The retriever is responsible for searching the most relevant chunks
#The retriver uses search methods, often based on vector similarity or keywords, to identify the passages or documents that most closely match your query
retriever = vector_db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,
        "fetch_k": 30,
        "lambda_mult": 0.7
    }
)

In [44]:
# INITIALIZE THE LLM

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1
)

SYSTEM_PROMPT = """
You are a retrieval assistant.

You MUST answer ONLY using the information contained in the provided transcripts.

Rules:
1. Do not use outside knowledge.
2. Do not infer information not explicitly stated.
3. If the answer is not found in the transcripts, reply:
   "This information is not available in the provided transcripts."
4. Use the transcripts intelligently by combining relevant passages.
"""

In [45]:
client = OpenAI()

def generate_diagram(topic):

    result = client.images.generate(
        model="gpt-image-1",
        prompt=f"educational diagram explaining {topic}, clean infographic style",
        size="1024x1024"
    )

    return result.data[0].url

In [40]:
# ==========================================================
# TOOL 1: SEARCH TRANSCRIPTS + IMAGE GENERATION
# ==========================================================
# This tool retrieves the most relevant transcript chunks
# from the vector database.

@tool
def search_transcripts(query: str) -> str:
    """
    Use this tool when the user asks for transcript content, details, facts,
    or wants to know what the videos say about a specific topic.
    """

    docs = retriever.get_relevant_documents(query)

    results = []
    for i, doc in enumerate(docs[:6], start=1):
        results.append(
            f"Result {i} | Video ID: {doc.metadata.get('video_id', 'unknown')}\n"
            f"{doc.page_content[:1200]}"
        )

    text_results = "\n\n".join(results)

    # Generate diagram/image from the user's query
    image_url = generate_diagram(query)

    final_output = f"""
Relevant transcript excerpts:

{text_results}

Generated diagram related to your question:
{image_url}
"""

    return final_output

In [29]:
# ==========================================================
# TOOL 2: SUMMARIZE VIDEO CONTENT
# ==========================================================
# This tool summarizes the transcript parts most relevant
# to the user's topic or question.


@tool
def summarize_video(topic: str) -> str:
    """
    Use this tool when the user asks for a summary of what the transcripted videos say
    about a topic.
    """

    docs = retriever.get_relevant_documents(topic)
    text = "\n\n".join([doc.page_content for doc in docs[:6]])

    prompt = f"""
You are summarizing content retrieved from several YouTube video transcripts.

Task:
1. Summarize only the information present in the retrieved text
2. Combine repeated ideas
3. If the retrieved text comes from multiple videos, synthesize them together
4. Do not invent missing details

Retrieved transcript content:
{text}
"""

    response = llm.invoke(prompt)
    return response.content

In [30]:
# ==========================================================
# TOOL 3: EXTRACT MAIN TOPICS
# ==========================================================
# This tool extracts the main topics discussed in the
# transcript sections related to the query.
@tool
def extract_topics(query: str) -> str:
    """
    Use this tool when the user asks for the main themes of the transcripted videos, general topics,
    or what the videos are mainly about.
    """

    docs = retriever.get_relevant_documents(query)
    text = "\n\n".join([doc.page_content for doc in docs[:8]])

    prompt = f"""
The following text comes from several YouTube video transcripts.

Your task:
- identify the broad topics shared across the retrieved videos
- ignore repeated details
- avoid focusing too much on one country, one anecdote, or one example
- produce 3 to 5 high-level themes

Return the answer as bullet points.

Retrieved text:
{text}
"""

    response = llm.invoke(prompt)
    return response.content


In [31]:
# ==========================================================
# TOOL 4: QUOTE EVIDENCE
# ==========================================================
# This tool returns supporting transcript excerpts
# for a specific question.

@tool
def quote_evidence(query: str) -> str:
    """
    Use this tool when the user asks for evidence, support, proof,
    or transcript excerpts about a specific claim or topic contained in the transcripts.
    """

    docs = retriever.get_relevant_documents(query)

    text = "\n\n".join([
        f"Video ID: {doc.metadata.get('video_id', 'unknown')}\n{doc.page_content[:1200]}"
        for doc in docs[:5]
    ])

    prompt = f"""
You are given transcript excerpts retrieved from the choosen videos about women.

YouTube videos.

Task:
- extract 3 short pieces of evidence relevant to the query
- keep them concise and readable
- for each piece of evidence, mention the Video ID
- only use the retrieved text
- if some retrieved text is irrelevant, ignore it

User query:
{query}

Retrieved transcript excerpts:
{text}
"""

    response = llm.invoke(prompt)
    return response.content

In [32]:
# ---------------------------------------------------------
# MEMORY
# ---------------------------------------------------------
# This allows the chatbot to remember previous messages
# in the conversation

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

In [41]:
# ---------------------------------------------------------
# CREATE THE AGENT
# ---------------------------------------------------------
# The agent can decide which tool to use depending
# on the user's question

tools = [
    search_transcripts,
    summarize_video,
    extract_topics,
    quote_evidence
]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=4
)

C:\Users\quinc\AppData\Local\Temp\ipykernel_10084\2038284788.py:14: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(


## TESTING AND EVALUATIONS

In [46]:
# ==========================================================
# SIMPLE TESTING AND EVALUATION
# ==========================================================
# We test the chatbot with a few sample questions.

test_questions = [
    "What are these videos mainly about?",
    "What political and religious topics appear across the videos?",
    "Give me transcript evidence about women's rights, religion, and power."
]

for q in test_questions:
    print("\nQUESTION:", q)
    answer = agent.run(q)
    print("ANSWER:", answer[:700])
    print("\n" + "="*80)


QUESTION: What are these videos mainly about?


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: extract_topics
Action Input: [Please provide the specific videos or transcripts you would like me to analyze for their main themes.]

C:\Users\quinc\AppData\Local\Temp\ipykernel_10084\3002827809.py:13: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  docs = retriever.get_relevant_documents(query)



Observation: - **Historical Context of Liberation Movements**: The discussions touch on the evolution of liberation concepts, particularly in the context of the Roaring 20s, highlighting how societal changes influence movements for freedom and rights.

- **Women's Suffrage and Empowerment**: The theme of women's rights, particularly suffrage, is prevalent, emphasizing the ongoing struggle for gender equality and the importance of recognizing women's contributions to history.

- **Cultural and Religious Interpretations**: There is a focus on the interpretation of religious texts and cultural practices, particularly in relation to women's rights and the distinction between spirituality and law, suggesting a need for reform and personal understanding within religious contexts.

- **Community Engagement and Storytelling**: The importance of community support and storytelling in shaping history and personal narratives is highlighted, encouraging viewers to share their experiences and persp

In [47]:
# TEST 2 USING LANGSMITH

test_questions = [
    "What are these videos mainly about?",
    "What political and religious topics appear across the videos?",
    "Give me transcript evidence about women's rights, religion, and power."
]

for q in test_questions:
    print("\nQUESTION:", q)
    print("="*80)

    answer = agent.invoke({"input": q})

    print("ANSWER:", answer["output"])
    print("\n" + "="*80)


QUESTION: What are these videos mainly about?


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? No
AI: The videos are mainly about several key themes: 

1. **Historical Context of Liberation Movements**: They explore the evolution of liberation concepts, particularly during the Roaring 20s, and how societal changes influence movements for freedom and rights.

2. **Women's Suffrage and Empowerment**: A significant focus is on women's rights, especially suffrage, highlighting the ongoing struggle for gender equality and the importance of recognizing women's contributions to history.

3. **Cultural and Religious Interpretations**: The videos discuss the interpretation of religious texts and cultural practices in relation to women's rights, emphasizing the need for reform and personal understanding within religious contexts.

4. **Community Engagement and Storytelling**: They underscore the importance of community support and storytelling in shaping history and

## We try a chat loop that utilize our transcripts

In [48]:
# ---------------------------------------------------------
# CHAT LOOP
# ---------------------------------------------------------
# User can ask questions about the YouTube video

while True:

    question = input("Ask about the video transcripts (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    docs = retriever.get_relevant_documents(question)

    if not docs:
        print("\nAnswer: This information is not available in the provided transcripts.")
        continue

    response = agent.run(question)

    print("\nAnswer:", response)

KeyboardInterrupt: Interrupted by user